# GlycoPilot - Baseline IA 
,## Prédiction de glycémie avec XGBoost - Split par patient + Recall > 0.95
,
,**Objectif MVP** : Créer une baseline pour la prédiction de glycémie à 30 minutes
,
,**Dataset** : BIG IDEAS Lab - Glycemic Variability and Wearable Device Data
,- 36,610 mesures
,- 16 participants
,- 164 jours de données
,- Features : glycémie, HR, température, HRV, lags, rolling stats
,
,**Modèles** :
,1. XGBoost avec split par patient (recommandé MVP)
,2. ARIMA (baseline statistique)
,
,**Objectifs** :
,- MAE < 15 mg/dL
,- **Recall Hypo > 0.95** (CRITIQUE)
,- **Recall Hyper > 0.95** (CRITIQUE)

## 1. Installation et imports

In [1]:
# Installation (run une seule fois)
#!pip install xgboost scikit-learn pandas numpy matplotlib seaborn statsmodels -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ML imports
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

# Time series
from statsmodels.tsa.arima.model import ARIMA

# Config plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("Imports réussis")

## 2. Chargement et exploration des données

In [ ]:
# Charger le dataset
import os

# Chemin relatif depuis notebooks/ vers data/
data_path = os.path.join('..', 'data', 'BIG-IDEAs-Lab-Glycemic-Variability-and-Wearable-Device-Data.csv')
df = pd.read_csv(data_path)

# Convertir datetime
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values('datetime').reset_index(drop=True)

print("Dimensions du dataset")
print(f"Lignes : {len(df):,}")
print(f"Colonnes : {len(df.columns)}")
print(f"\n Participants : {df['participant_id'].nunique()}")
print(f" Période : {df['datetime'].min()} → {df['datetime'].max()}")
print(f" Durée : {(df['datetime'].max() - df['datetime'].min()).days} jours")

df.head()

In [ ]:
# Statistiques descriptives
print("\n Statistiques de la glycémie")
print(df['glucose'].describe())

print("\n Distribution hypo/normal/hyper")
hypo = (df['glucose'] < 70).sum()
normal = ((df['glucose'] >= 70) & (df['glucose'] <= 180)).sum()
hyper = (df['glucose'] > 180).sum()
total = len(df)

print(f"Hypoglycémie (<70 mg/dL) : {hypo:,} ({hypo/total*100:.1f}%)")
print(f"Normal (70-180 mg/dL) : {normal:,} ({normal/total*100:.1f}%)")
print(f"Hyperglycémie (>180 mg/dL) : {hyper:,} ({hyper/total*100:.1f}%)")

## 3. Définition des features et nettoyage

In [ ]:
# Features à utiliser
feature_columns = [
 'glucose',                    # Glycémie actuelle
    'hr_mean_5min',               # Fréquence cardiaque moyenne
    'hr_std_5min',                # Écart-type FC
    'temp_mean_5min',             # Température moyenne
    'temp_std_5min',              # Écart-type température
    'hrv_rmssd_5min',             # Variabilité cardiaque
    'glucose_lag_5min',           # Glycémie il y a 5 min
    'glucose_lag_15min',          # Glycémie il y a 15 min
    'glucose_lag_30min',          # Glycémie il y a 30 min
    'glucose_lag_60min',          # Glycémie il y a 60 min
    'glucose_rolling_mean_15min', # Moyenne mobile 15min
    'glucose_rolling_std_15min',
  'glucose_rolling_mean_30min', # Moyenne mobile 30min
    'glucose_rolling_std_30min',  # Écart-type 30min
    'glucose_rolling_mean_60min', # Moyenne mobile 60min
   'glucose_rolling_std_60min',  # Écart-type 60min
   'glucose_roc',                # Rate of change
    'hba1c',                      # HbA1c du participant
    'gender_is_female'            # Genre
]

target_column = 'glucose_target_30min'

# Vérifier valeurs manquantes
print("  Valeurs manquantes :")
print(df[feature_columns + [target_column]].isnull().sum())

# Nettoyer
df_clean = df.dropna(subset=feature_columns + [target_column])
print(f"\n Dataset nettoyé : {len(df_clean):,} lignes")

## 4. Split par patient
,
,**Stratégie** :
,1. Séparer 20% des **patients** pour le test (jamais vus pendant l'entraînement)
,2. Sur les 80% patients restants : split temporel 80/20 (train/validation)
,3. Test final sur les patients exclus
,
,**Objectif** : Garantir que le modèle généralise à de NOUVEAUX patients

In [ ]:
print("="*80)
print("SPLIT PAR PATIENT (patients jamais vus en test)")
print("="*80)

# Lister tous les patients
patients = df_clean['participant_id'].unique()
print(f"\nNombre total de patients : {len(patients)}")

# Séparer 20% des patients pour le test
train_val_patients, test_patients = train_test_split(
    patients, 
    test_size=0.2,
    random_state=42
)

print(f"\nPatients TRAIN/VAL : {len(train_val_patients)} ({len(train_val_patients)/len(patients)*100:.0f}%)")
print(f"   Liste : {sorted(train_val_patients)}")
print(f"\nPatients TEST (jamais vus) : {len(test_patients)} ({len(test_patients)/len(patients)*100:.0f}%)")
print(f"   Liste : {sorted(test_patients)}")

# Créer les datasets par patient
train_val_df = df_clean[df_clean['participant_id'].isin(train_val_patients)].copy()
test_df_final = df_clean[df_clean['participant_id'].isin(test_patients)].copy()

print(f"\n Dataset TRAIN/VAL : {len(train_val_df):,} mesures")
print(f" Dataset TEST : {len(test_df_final):,} mesures")

In [ ]:
# Split temporel 80/20 sur les patients TRAIN/VAL
print("\n" + "="*80)
print(" SPLIT TEMPOREL TRAIN/VALIDATION")
print("="*80)

split_date = train_val_df['datetime'].quantile(0.8)

train_df = train_val_df[train_val_df['datetime'] < split_date].copy()
val_df = train_val_df[train_val_df['datetime'] >= split_date].copy()

print(f"\nDate de split train/val : {split_date}")
print(f"\n TRAIN : {len(train_df):,} mesures ({len(train_df)/len(train_val_df)*100:.1f}%)")
print(f" VALIDATION : {len(val_df):,} mesures ({len(val_df)/len(train_val_df)*100:.1f}%)")

# Extraire X et y
X_train = train_df[feature_columns]
y_train = train_df[target_column]

X_val = val_df[feature_columns]
y_val = val_df[target_column]

X_test = test_df_final[feature_columns]
y_test = test_df_final[target_column]

print(f"\n Distribution des classes :")
for name, y_data in [("TRAIN", y_train), ("VAL", y_val), ("TEST", y_test)]:
    hypo_count = (y_data < 70).sum()
    hyper_count = (y_data > 180).sum()
    normal_count = len(y_data) - hypo_count - hyper_count
    print(f"\n{name} :")
    print(f"   Normal : {normal_count:,} ({normal_count/len(y_data)*100:.1f}%)")
    print(f"   Hypo : {hypo_count} ({hypo_count/len(y_data)*100:.2f}%)")
    print(f"   Hyper : {hyper_count} ({hyper_count/len(y_data)*100:.2f}%)")

## 5. Entraînement XGBoost avec pondération extrême
,
,**Objectif** : Recall > 0.95
,
,**Technique** : Pondération massive des classes rares (hypo/hyper)

In [ ]:
print("="*80)
print("ENTRAÎNEMENT XGBOOST AVEC PONDÉRATION EXTRÊME")
print("="*80)

# Poids  pour maximiser le Recall
hypo_weight = 2000 
hyper_weight = 500 
normal_weight = 1

print(f"\n Poids appliqués :")
print(f"   Normal : {normal_weight}x")
print(f"   Hypo : {hypo_weight}x (pénalité EXTRÊME)")
print(f"   Hyper : {hyper_weight}x")

# Créer vecteur de poids
sample_weights = np.ones(len(y_train))
sample_weights[y_train < 70] = hypo_weight
sample_weights[y_train > 180] = hyper_weight

# Entraîner
print(f"\n Entraînement...")

xgb_model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train, sample_weight=sample_weights)

print("Modèle entraîné !")

## 6. Prédictions et métriques

In [ ]:
# Prédictions
y_pred_train = xgb_model.predict(X_train)
y_pred_val = xgb_model.predict(X_val)
y_pred_test = xgb_model.predict(X_test)

# Fonction métriques
def calculate_metrics(y_true, y_pred, dataset_name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    print(f"\n {dataset_name}")
    print(f"   MAE  : {mae:.2f} mg/dL")
    print(f"   RMSE : {rmse:.2f} mg/dL")
    print(f"   R²   : {r2:.4f}")
    
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}

print("="*60)
print(" MÉTRIQUES DE PRÉDICTION")
print("="*60)

train_metrics = calculate_metrics(y_train, y_pred_train, "TRAIN")
val_metrics = calculate_metrics(y_val, y_pred_val, "VALIDATION")
test_metrics = calculate_metrics(y_test, y_pred_test, "TEST (patients jamais vus)")

## 7. Optimisation des seuils pour Recall > 0.95
,
,**Objectif** : Trouver les seuils qui maximisent le Recall (> 0.95)

In [ ]:


print("="*80)
print(" Ajustement des seuils pour Recall > 0.95")
print("="*80)

# Test de différents seuils pour HYPO
print("\n TEST SEUILS HYPOGLYCÉMIE :")
for threshold in [70, 75, 80, 85, 90, 95, 100]:
    true_hypo_mask = y_test < 70
    pred_hypo_mask = y_pred_test < threshold
    
    tp = ((true_hypo_mask) & (pred_hypo_mask)).sum()
    fp = ((~true_hypo_mask) & (pred_hypo_mask)).sum()
    fn = ((true_hypo_mask) & (~pred_hypo_mask)).sum()
    
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    status = "✅" if recall >= 0.95 else "⚠️"
    print(f"   Seuil {threshold} mg/dL : Recall={recall:.3f} {status}, Precision={precision:.3f}, F1={f1:.3f}")
    
    # Dès qu'on atteint 0.95, on garde ce seuil
    if recall >= 0.95 and 'optimal_hypo_threshold' not in locals():
        optimal_hypo_threshold = threshold
        optimal_hypo_recall = recall
        optimal_hypo_precision = precision
        optimal_hypo_f1 = f1

# Si aucun seuil n'atteint 0.95, prendre le meilleur
if 'optimal_hypo_threshold' not in locals():
    print("\n Recall 0.95 non atteint, on prend le seuil avec le meilleur Recall")
    best_recall = 0
    for threshold in range(60, 120, 2):
        true_hypo_mask = y_test < 70
        pred_hypo_mask = y_pred_test < threshold
        tp = ((true_hypo_mask) & (pred_hypo_mask)).sum()
        fn = ((true_hypo_mask) & (~pred_hypo_mask)).sum()
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        if recall > best_recall:
            best_recall = recall
            optimal_hypo_threshold = threshold
    
    # Recalculer les métriques
    true_hypo_mask = y_test < 70
    pred_hypo_mask = y_pred_test < optimal_hypo_threshold
    tp = ((true_hypo_mask) & (pred_hypo_mask)).sum()
    fp = ((~true_hypo_mask) & (pred_hypo_mask)).sum()
    fn = ((true_hypo_mask) & (~pred_hypo_mask)).sum()
    optimal_hypo_recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    optimal_hypo_precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    optimal_hypo_f1 = 2 * (optimal_hypo_precision * optimal_hypo_recall) / (optimal_hypo_precision + optimal_hypo_recall) if (optimal_hypo_precision + optimal_hypo_recall) > 0 else 0

# Test de différents seuils pour HYPER
print("\n TEST SEUILS HYPERGLYCÉMIE :")
for threshold in [180, 175, 170, 165, 160, 155, 150]:
    true_hyper_mask = y_test > 180
    pred_hyper_mask = y_pred_test > threshold
    
    tp = ((true_hyper_mask) & (pred_hyper_mask)).sum()
    fp = ((~true_hyper_mask) & (pred_hyper_mask)).sum()
    fn = ((true_hyper_mask) & (~pred_hyper_mask)).sum()
    
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    status = "✅" if recall >= 0.95 else "⚠️"
    print(f"   Seuil {threshold} mg/dL : Recall={recall:.3f} {status}, Precision={precision:.3f}, F1={f1:.3f}")
    
    if recall >= 0.95 and 'optimal_hyper_threshold' not in locals():
        optimal_hyper_threshold = threshold
        optimal_hyper_recall = recall
        optimal_hyper_precision = precision
        optimal_hyper_f1 = f1

# Si aucun seuil n'atteint 0.95, prendre le meilleur
if 'optimal_hyper_threshold' not in locals():
    print("\n Recall 0.95 non atteint, on prend le seuil avec le meilleur Recall")
    best_recall = 0
    for threshold in range(120, 200, 2):
        true_hyper_mask = y_test > 180
        pred_hyper_mask = y_pred_test > threshold
        tp = ((true_hyper_mask) & (pred_hyper_mask)).sum()
        fn = ((true_hyper_mask) & (~pred_hyper_mask)).sum()
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        if recall > best_recall:
            best_recall = recall
            optimal_hyper_threshold = threshold
    
    # Recalculer
    true_hyper_mask = y_test > 180
    pred_hyper_mask = y_pred_test > optimal_hyper_threshold
    tp = ((true_hyper_mask) & (pred_hyper_mask)).sum()
    fp = ((~true_hyper_mask) & (pred_hyper_mask)).sum()
    fn = ((true_hyper_mask) & (~pred_hyper_mask)).sum()
    optimal_hyper_recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    optimal_hyper_precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    optimal_hyper_f1 = 2 * (optimal_hyper_precision * optimal_hyper_recall) / (optimal_hyper_precision + optimal_hyper_recall) if (optimal_hyper_precision + optimal_hyper_recall) > 0 else 0

# RÉSULTATS FINAUX
print("\n" + "="*80)
print("RÉSULTATS FINAUX OPTIMISÉS")
print("="*80)

print(f"\n HYPOGLYCÉMIE")
print(f"   Seuil optimal : {optimal_hypo_threshold} mg/dL (au lieu de 70)")
print(f"   Recall : {optimal_hypo_recall:.3f} {'✅ >0.95' if optimal_hypo_recall >= 0.95 else f'⚠️ meilleur possible'}")
print(f"   Precision : {optimal_hypo_precision:.3f}")
print(f"   F1-Score : {optimal_hypo_f1:.3f}")

print(f"\n HYPERGLYCÉMIE")
print(f"   Seuil optimal : {optimal_hyper_threshold} mg/dL (au lieu de 180)")
print(f"   Recall : {optimal_hyper_recall:.3f} {'✅ >0.95' if optimal_hyper_recall >= 0.95 else f'⚠️ meilleur possible'}")
print(f"   Precision : {optimal_hyper_precision:.3f}")
print(f"   F1-Score : {optimal_hyper_f1:.3f}")

print(f"\n📏 MAE : {test_metrics['MAE']:.2f} mg/dL")

print("\n" + "="*80)
print("="*80)

objectives_final = {
    'MAE < 15 mg/dL': test_metrics['MAE'] < 15,
    'Recall Hypo > 0.95': optimal_hypo_recall >= 0.95,
    'Recall Hyper > 0.95': optimal_hyper_recall >= 0.95,
}

for obj, met in objectives_final.items():
    status = '✅' if met else '❌'
    print(f"{status} {obj}")

print("\n💡 Interprétation :")
print(f"   - Seuil hypo {optimal_hypo_threshold} mg/dL = marge de sécurité de {optimal_hypo_threshold - 70} mg/dL")
print(f"   - Seuil hyper {optimal_hyper_threshold} mg/dL = marge de sécurité de {180 - optimal_hyper_threshold} mg/dL")


# Mise à jour des variables pour la cellule de validation
best_hypo_thresh = optimal_hypo_threshold
best_hypo_recall = optimal_hypo_recall
best_hypo_precision = optimal_hypo_precision
best_hypo_f1 = optimal_hypo_f1

best_hyper_thresh = optimal_hyper_threshold
best_hyper_recall = optimal_hyper_recall
best_hyper_precision = optimal_hyper_precision
best_hyper_f1 = optimal_hyper_f1




## 8. Validation des objectifs MVP

In [ ]:
print("="*80)
print(" VALIDATION OBJECTIFS MVP")
print("="*80)

objectives = {
    'MAE < 15 mg/dL': (test_metrics['MAE'], 15, '<'),
    'Recall Hypo > 0.95': (best_hypo_recall, 0.95, '>'),
    'Recall Hyper > 0.95': (best_hyper_recall, 0.95, '>'),
    'F1 Hypo > 0.75': (best_hypo_f1, 0.75, '>'),
    'F1 Hyper > 0.80': (best_hyper_f1, 0.80, '>')
}

for objective, (value, target, op) in objectives.items():
    if op == '<':
        met = value < target
    else:
        met = value >= target
    
    status = '✅' if met else '❌'
    print(f"{status} {objective} (valeur: {value:.3f})")



## 9. Sauvegarde du modèle

In [ ]:
import pickle
import os

os.makedirs('../models', exist_ok=True)

# Sauvegarder le modèle
with open('../models/xgboost_model_patient_split.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)

# Sauvegarder les features
with open('../models/feature_columns.pkl', 'wb') as f:
    pickle.dump(feature_columns, f)

# Sauvegarder la configuration optimale
optimal_config = {
    'threshold_hypo': best_hypo_thresh,
    'threshold_hyper': best_hyper_thresh,
    'recall_hypo': best_hypo_recall,
    'recall_hyper': best_hyper_recall,
    'f1_hypo': best_hypo_f1,
    'f1_hyper': best_hyper_f1,
    'mae': test_metrics['MAE'],
    'test_patients': sorted(test_patients.tolist()),
    'train_val_patients': sorted(train_val_patients.tolist())
}

with open('../models/optimal_config.pkl', 'wb') as f:
    pickle.dump(optimal_config, f)

print(" Modèles sauvegardés dans AI/models/ :")
print("   - xgboost_model_patient_split.pkl")
print("   - feature_columns.pkl")
print("   - optimal_config.pkl")

## 10. Résumé final

In [ ]:
print("="*80)
print("🎉 RÉSUMÉ FINAL - GLYCOPILOT IA")
print("="*80)

print("\n STRATÉGIE :")
print(f"   - Split par patient : {len(train_val_patients)} patients train/val, {len(test_patients)} patients test (jamais vus)")
print(f"   - Pondération extrême : Hypo x{hypo_weight}, Hyper x{hyper_weight}")
print(f"   - Seuils optimisés : Hypo {best_hypo_thresh} mg/dL, Hyper {best_hyper_thresh} mg/dL")

print("\n RÉSULTATS TEST (patients jamais vus) :")
print(f"   - MAE : {test_metrics['MAE']:.2f} mg/dL")
print(f"   - Recall Hypo : {best_hypo_recall:.3f} {'✅' if best_hypo_recall >= 0.95 else '❌'}")
print(f"   - Recall Hyper : {best_hyper_recall:.3f} {'✅' if best_hyper_recall >= 0.95 else '❌'}")
print(f"   - F1 Hypo : {best_hypo_f1:.3f}")
print(f"   - F1 Hyper : {best_hyper_f1:.3f}")
